# 02. Baseline (Bab 5 dan Minggu 2)
Dua bagian: (1) reproduksi protokol baseline repo referensi G1 (`src/reproduce_g1.py`), (2) baseline kita sendiri pada skenario A (split acak, semua fitur) dari `results/metrics.csv`. Angka README G1 tidak disalin ke sini.

In [1]:
import os, sys
from pathlib import Path
os.chdir(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd())
sys.path.insert(0, 'src')
import json
import pandas as pd
pd.set_option('display.width', 160)
print('akar repositori:', Path.cwd().name)

akar repositori: ttki-phishing-detection


## 1. Reproduksi protokol G1 pada berkas data kita
SHA-256 `Dataset.csv` G1 identik dengan berkas kita (`a236549c...d06d1c6`). Protokol: 50 fitur numerik, split stratified 70:30 seed 42, tanpa deduplikasi, model tanpa seed.

In [2]:
g = json.load(open('results/reproduksi_g1.json', encoding='utf-8'))
print(g['lingkungan'], g['protokol'])
rows = []
for m, v in g['hasil'].items():
    for pos in ('positif_label1', 'positif_phishing'):
        rows.append({'model': m, 'kelas positif': pos, **{k: round(x, 6) for k, x in v[pos].items()}})
pd.DataFrame(rows)

{'python': '3.11.9', 'scikit_learn': '1.9.1'} {'baris': 235795, 'n_fitur': 50, 'n_train': 165056, 'n_test': 70739, 'catatan': 'model tanpa random_state seperti G1; hasil DT/RF/ET dapat berbeda tipis antar run'}


,model,kelas positif,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression,positif_label1,0.999901,0.999901,0.999926,0.999913,0.999999
1,Logistic Regression,positif_phishing,0.999901,0.999901,0.999868,0.999884,0.999999
2,Decision Tree,positif_label1,1.000000,1.000000,1.000000,1.000000,1.000000
3,Decision Tree,positif_phishing,1.000000,1.000000,1.000000,1.000000,1.000000
4,Random Forest,positif_label1,1.000000,1.000000,1.000000,1.000000,1.000000
5,Random Forest,positif_phishing,1.000000,1.000000,1.000000,1.000000,1.000000
6,Extra Trees,positif_label1,0.999972,0.999951,1.000000,0.999975,1.000000
7,Extra Trees,positif_phishing,0.999972,1.000000,0.999934,0.999967,1.000000


## 2. Baseline kita: skenario A (split acak 70:15:15, 50 fitur), test, lima seed

In [3]:
m = pd.read_csv('results/metrics.csv')
t = m[(m.skenario=='A') & (m.bagian=='test')]
t.groupby('model')[['f1','precision','recall','roc_auc','fn']].agg(['mean','std']).round(5)

f1          precision        recall          roc_auc            fn         
          mean      std      mean  std     mean      std    mean      std mean      std
model                                                                                  
LR     0.99989  0.00004       1.0  0.0  0.99979  0.00009     1.0  0.00001  3.2  1.30384
RF     0.99999  0.00001       1.0  0.0  0.99999  0.00003     1.0  0.00000  0.2  0.44721